<a href="https://colab.research.google.com/github/Thawin2551/python/blob/main/FahMai_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Starter Kit FahMai RAG

This notebook walks through building a minimalist **Retrieval-Augmented Generation (RAG)** system for the FahMai electronics store Q&A challenge.

**Sections:**
- **Section 0:** Setup & LLM Test
- **Section 1:** Dense Retrieval (MiniLM)
- **Section 2:** Sparse Retrieval (BM25)
- **Section 3:** Hybrid Retrieval (RRF)
- **Section 4:** What's Next?

In [1]:
# Load the data from kaggle and upload here.
!unzip data-fahmai.zip

Archive:  data-fahmai.zip
   creating: data-fahmai/
   creating: data-fahmai/knowledge_base/
   creating: data-fahmai/knowledge_base/policies/
  inflating: data-fahmai/knowledge_base/policies/cancellation_policy.md  
  inflating: data-fahmai/knowledge_base/policies/membership_points_policy.md  
  inflating: data-fahmai/knowledge_base/policies/return_policy.md  
  inflating: data-fahmai/knowledge_base/policies/shipping_policy.md  
  inflating: data-fahmai/knowledge_base/policies/warranty_policy.md  
   creating: data-fahmai/knowledge_base/products/
  inflating: data-fahmai/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data-fahmai/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data-fahmai/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data-fahmai/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data-fahmai/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data-fah

In [2]:
# === CONFIGURATION ===
# Change N_QUESTIONS to 100 for a full competition run.

N_QUESTIONS = 10  # 10 for demo, 100 for real submission
DATA_DIR = "/content/data-fahmai"
KB_DIR = f"{DATA_DIR}/knowledge_base"

# Checking Question.csv

In [12]:
import pandas as pd
pd.read_csv("/content/data-fahmai/questions.csv")

,id,question,choice_1,choice_2,choice_3,choice_4,choice_5,choice_6,choice_7,choice_8,choice_9,choice_10
0,1,Watch S3 Ultra กันน้ำได้กี่ ATM ครับ,3 ATM,IP68,5 ATM,IP67,10 ATM,20 ATM,IPX8,1 ATM,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,"2,990 บาท","4,490 บาท","1,990 บาท","4,990 บาท","3,490 บาท","2,490 บาท","3,990 บาท","5,490 บาท",ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,Bluetooth 5.0,Bluetooth 5.3,Bluetooth 5.4,Bluetooth 4.2,Bluetooth 5.2,Bluetooth 5.1,Bluetooth 4.0,Bluetooth 5.5,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
3,4,อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ยนเป็นเค...,ได้ ลดราคาสูงสุด 30%,ได้ เฉพาะสมาร์ทโฟนสายฟ้าเท่านั้น,ได้ ต้องเป็นสมาชิก Gold ขึ้นไป,ได้ เฉพาะสินค้าที่ซื้อจากฟ้าใหม่,"ได้ แต่ต้องซื้อเครื่องใหม่ราคาเกิน 10,000 บาท",ไม่มีบริการ Trade-in ในขณะนี้,ได้ เฉพาะช่วง Mega Sale,ได้ ต้องนำไปที่สาขาเท่านั้น,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
4,5,สั่งของจากฟ้าใหม่ สั่งได้ครั้งละกี่ชิ้นครับ,ไม่จำกัดจำนวน,5 รายการ,20 รายการ,15 รายการ,3 รายการ,10 รายการ,50 รายการ,8 รายการ,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,ซื้อแท่นชาร์จ PulseGear ChargePad 15W มา อยากว...,ได้ครับ ChargePad 15W ชาร์จอุปกรณ์ Qi ทุกรุ่น ...,ได้ แต่ชาร์จช้ามาก เพราะเคสหูฟังรับไฟได้แค่ 5W,ได้ครับ วางเคสหูฟัง NovaBuds Pro บนแท่น Charge...,ไม่ได้ ChargePad 15W ชาร์จได้เฉพาะสมาร์ทโฟน ไม...,ไม่ได้ เคส NovaBuds Pro ชาร์จผ่าน USB-C เท่านั...,ได้ แต่ต้องอัปเดตเฟิร์มแวร์เคสหูฟังก่อน,ไม่ได้ ต้องซื้อ ChargePad รุ่น Pro ที่รองรับ T...,ได้ แต่ต้องถอดหูฟังออกจากเคสก่อนชาร์จ,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
96,97,ซื้อ ArcWave SoundPillar 300 มา ทำตกพื้นเสียหา...,เคลมประกันได้ครับ ArcWave มีประกัน 18 เดือนครอ...,เคลมประกันไม่ได้ แต่ซื้อ Care+ คุ้มครองอุบัติเ...,เคลมประกันได้ครับ ArcWave SoundPillar 300 มีปร...,ซื้อ Care+ เพิ่มได้ ฿590 ต่อปี คุ้มครองตกกระแทก,เคลม Care+ ได้ จ่ายค่าซ่อมส่วนต่าง 30% ของราคา...,ต้องส่งซ่อมที่ศูนย์ ArcWave โดยตรง ไม่ผ่านฟ้าใหม่,ไม่ได้ทั้งสองทาง ประกันปกติไม่คุ้มครองอุบัติเห...,ไม่ได้ แต่ซื้อประกันเพิ่มจากบริษัทประกันภัยภาย...,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
97,98,"สั่งซื้อ SoundBar Pro 500 ราคา ฿15,990 จัดส่งไ...",ฟรี เพราะมูลค่าสั่งซื้อเกิน ฿500,฿200 เฉพาะค่าจัดส่งสินค้าหนักเกิน 30 กก. เพียง...,฿300 คือค่าสินค้าหนัก ฿200 + ค่าขนขึ้นชั้น 6 อ...,฿400 คือค่าสินค้าหนัก ฿200 + ค่าขนขึ้นชั้น 5 แ...,"฿500 คือค่าสินค้าหนัก ฿200 + ค่าขนชั้น 4, 5, 6...",฿800 คือค่าสินค้าหนัก ฿200 + ค่าขนทุกชั้น ฿100...,฿100 ค่าขนขึ้นชั้นอย่างเดียว เพราะจัดส่งฟรีรวม...,฿350 คือค่าจัดส่งมาตรฐาน ฿50 + ค่าสินค้าหนัก ฿...,ไม่มีข้อมูลเพียงพอในฐานข้อมูลสำหรับคำนวณค่าจัดส่ง,คำถามนี้ไม่เกี่ยวข้องกับร้านฟ้าใหม่
98,99,"สั่ง PulseGear Power Bank 30,000 mAh ส่งไปเกาะ...",1-2 วันทำการ ถ้าสั่งก่อนเที่ยงวัน,3-5 วันทำการ ตามตารางจัดส่งภาคใต้,5-7 วันทำการ ตามตารางจัดส่งพื้นที่เกาะ,8-12 วันทำการ เพราะเป็นเกาะ (5-7 วัน) + ส่งทาง...,"10-14 วันทำการ เนื่องจากแบตสำรองเกิน 20,000 mA...","ไม่สามารถจัดส่งได้ เพราะ Power Bank เกิน 20,00...",15-20 วันทำการ เพราะต้องส่งเป็นพัสดุพิเศษทางเรือ,2-3 วันทำการ ถ้าเลือก Express,ไม่มีข้อมูลเพียงพอในฐานข้อมูลสำหรับตอบคำถามนี้,คำถามนี้ไม่เกี่ยวข้องกับร้านฟ้าใหม่


---
## Section 0: Setup & LLM Test

First, install dependencies and test the ThaiLLM API — **without** any retrieval. This shows why RAG is needed.

ThaiLLM website: https://playground.thaillm.or.th/chat/

In [3]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 76.8 MB/s eta 0:00:00


In [4]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from google.colab import userdata

THAILLM_API_KEY = userdata.get('THAILLM_API_KEY')

In [ ]:
def ask_llm(messages, model="typhoon", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            # Rate limit — wait and retry
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None


def parse_answer(text):
    """Extract answer number from LLM response."""
    if text is None:
        return None
    # Remove any <think>...</think> blocks (some models do chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Look for ANSWER: X pattern
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        return int(m.group(1))
    # Fallback: first standalone number 1-10
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    return None

### Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [ ]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

LLM response (no context): Watch S3 Ultra ของ Samsung ที่เปิดตัวในปี 2023 นั้น **กันน้ำได้ระดับ 5 ATM** หรือเทียบเท่ากับ **50 เมตร** ตามมาตรฐาน IPX8 ที่ระบุไว้ในข้อมูลผลิตภัณฑ์

ซึ่งหมายความว่า:

- สามารถกันน้ำได้ในระดับความลึกสูงสุด 50 เมตร
- สามารถใช้งานได้ในสภาพแวดล้อมที่มีน้ำ เช่น ว่ายน้ำ หรือล้างมือ
- แต่ไม่ควรใช้ในกิจกรรมที่มีแรงดันน้ำสูง เช่น ดำน้ำลึก หรือใช้ในน้ำแรงดันสูง

อย่างไรก็ตาม ควรตรวจสอบข้อมูลจากเว็บไซต์ทางการของ Samsung หรือคู่มือการใช้งานเพื่อความแน่ใจ เพราะข้อมูลอาจมีการอัปเดตหรือเปลี่ยนแปลงได้ตามรุ่นย่อยหรือการผลิต

สรุป: **Watch S3 Ultra กันน้ำได้ 5 ATM (50 เมตร)** ครับ ✅

→ The LLM doesn't know FahMai-specific facts. We need RAG!


### Load Questions

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS} for this run")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in questions[0]["choices"].items():
    print(f"  {k}. {v}")

Loaded 100 questions, using first 10 for this run

Example — Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  1. 3 ATM
  2. IP68
  3. 5 ATM
  4. IP67
  5. 10 ATM
  6. 20 ATM
  7. IPX8
  8. 1 ATM
  9. ไม่มีข้อมูลนี้ในฐานข้อมูล
  10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


---
## Section 1: Dense Retrieval (MiniLM)

**Dense retrieval** converts text into vectors (embeddings) and finds relevant chunks by cosine similarity.

The pipeline: **Load docs → Chunk → Embed → Retrieve → Generate**

### 1.1 Load Knowledge Base

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


### 1.2 Chunking

LLMs have limited context windows, and long documents dilute relevance. We split each document into smaller **chunks** using a fixed-size sliding window with overlap.

In [ ]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

def make_chunks(text, size, overlap):
    """Split text into overlapping fixed-size windows."""
    if len(text) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return chunks

# Build all chunks
chunks = []
for doc in documents:
    for window in make_chunks(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        chunks.append({"text": window, "source": doc["path"]})

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\n--- Sample chunk ---")
print(f"Source: {chunks[0]['source']}")
print(chunks[0]["text"][:300])

Created 1055 chunks from 118 documents

--- Sample chunk ---
Source: policies/cancellation_policy.md
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้


### 1.3 Embedding

We use `paraphrase-multilingual-MiniLM-L12-v2` — a small (118M params), multilingual embedding model that produces 384-dimensional vectors. It supports Thai out of the box and doesn't require any special prefixes.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Embed all chunks
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")  # (n_chunks, 384)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Chunk embeddings shape: (1055, 384)


### 1.4 Retrieve

Embed the question, then find the most similar chunks via dot product (= cosine similarity for normalized vectors).

In [ ]:
TOP_K = 5

def dense_retrieve(query, chunk_embs, k=TOP_K):
    """Return indices of top-k most similar chunks to the query."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Demo: retrieve for Q1
q = questions[0]
idx, scores = dense_retrieve(q["question"], chunk_embeddings)

print(f"Question: {q['question']}\n")
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"  Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")
    print(f"  {chunks[i]['text'][:150]}...")
    print()

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

  Rank 1 (score=0.675) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  ือน** ผ่านบัตรเครดิตที่ร่วมรายการ
(หมดเขต 30 มิถุนายน 2569)

ตัวอย่าง: ฿14,990 ÷ 6 เดือน = เพียง ฿2,498.33/เดือน

---

## คำถามที่พบบ่อยเกี่ยวกับสินค้...

  Rank 2 (score=0.626) [products/WK-SW-003_wongkhojon_watch_s3.md]
  ATM ในราคาที่เข้าถึงได้ง่ายกว่า S3 Pro กว่าครึ่ง

ตัวเรือนอะลูมิเนียมอัลลอยด์เคลือบผิวแบบ anodized ให้ความทนทานและน้ำหนักเบาพร้อมกัน หน้าจอ AMOLED 1.7...

  Rank 3 (score=0.602) [products/WK-SW-002_wongkhojon_watch_s3_pro.md]
  - **NFC Pay:** รองรับการชำระเงินผ่าน FahMai Pay ที่เครื่องรับที่สนับสนุน NFC

---

## โปรโมชันปัจจุบัน

**รับ FahMai Points x2** เมื่อซื้อ Watch S3 Pr...

  Rank 4 (score=0.601) [products/WK-SW-004_wongkhojon_watch_s3_se.md]
  าบสำคัญ:** Watch S3 SE ไม่มี GPS ในตัว (ใช้ GPS จากโทรศัพท์แทน — Connected GPS) ไม่มี ECG และไม่มี NFC Pay มาตรฐานกันน้ำ 3 ATM (30 เมตร) รองรับฝนและกา...

  Rank 5 (score=0.597) [products/WK-SW-003_

### 1.5 Generate Answer

Send the retrieved context + question + choices to the LLM and parse the answer.

In [ ]:
# A very basic system prompt — you should improve this!
SYSTEM_PROMPT = "ตอบคำถามจากข้อมูลที่ให้มา เลือกตัวเลือกที่ถูกต้องที่สุด ตอบเป็น ANSWER: X เท่านั้น"

def build_rag_prompt(question, choices, retrieved_chunks):
    """Build the user prompt with retrieved context.

    TODO: Design your own prompt format.
    Think about: How should you present the context? The choices?
    What instructions help the LLM pick the right answer?
    """
    context = "\n\n".join(
        f"--- Chunk {i+1} ---\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())
    # === CUSTOMIZE THIS PROMPT ===
    return (
        f"{context}\n\n"
        f"คำถาม: {question}\n\n"
        f"ตัวเลือก:\n{choices_text}\n\n"
        f"ตอบ ANSWER: X (X คือหมายเลขตัวเลือก 1-10)"
    )

# Demo: answer Q1
q = questions[0]
idx, _ = dense_retrieve(q["question"], chunk_embeddings)
retrieved = [chunks[i] for i in idx]

prompt = build_rag_prompt(q["question"], q["choices"], retrieved)
raw = ask_llm([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
])
answer = parse_answer(raw)
print(f"Q{q['id']}: {q['question']}")
print(f"LLM raw: {raw}")
print(f"Parsed answer: {answer}")

Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
LLM raw: ANSWER: 5
Parsed answer: 5


In [ ]:
retrieved

[{'text': 'ือน** ผ่านบัตรเครดิตที่ร่วมรายการ\n(หมดเขต 30 มิถุนายน 2569)\n\nตัวอย่าง: ฿14,990 ÷ 6 เดือน = เพียง ฿2,498.33/เดือน\n\n---\n\n## คำถามที่พบบ่อยเกี่ยวกับสินค้านี้\n\n**Q: Watch S3 Ultra ต่างจาก Watch S3 Pro ยังไงบ้างครับ?**\nA: S3 Ultra แตกต่างจาก S3 Pro ในหลายด้านครับ ได้แก่ (1) ตัวเรือนไทเทเนียม เบากว่าสแตนเลสของ S3 Pro, (2) กันน้ำ 100 เมตร พร้อม Dive Mode — S3 Pro กันน้ำ 50 เมตรแต่ไม่มี Dive Mode, (3) GPS แบบ Dual-Band แม่นยำกว่า S3 Pro ที่เป็น Single-Band, (4) หน้าจอใหญ่กว่า 1.9 นิ้ว vs 1.7 นิ้ว, (5) แบตเตอรี่อึดก',
  'source': 'products/WK-SW-001_wongkhojon_watch_s3_ultra.md'},
 {'text': 'ATM ในราคาที่เข้าถึงได้ง่ายกว่า S3 Pro กว่าครึ่ง\n\nตัวเรือนอะลูมิเนียมอัลลอยด์เคลือบผิวแบบ anodized ให้ความทนทานและน้ำหนักเบาพร้อมกัน หน้าจอ AMOLED 1.7 นิ้วที่มีความสว่าง 1,000 nits ใช้งานได้แม้กลางแดด สีสันสดใส ประหยัดพลังงาน ทำให้แบตเตอรี่อยู่ได้นานถึง 18 ชั่วโมงในโหมดใช้งานเต็มรูปแบบ หรือสูงสุด 10 วันในโหมดประหยัด\n\nGPS single-band รองรับดาวเทียม GPS และ GLONASS ติดตามเส้นทางการออก

### 1.6 Run All Questions (Dense)

Loop through questions, retrieve, generate, and collect predictions.

In [ ]:
def run_pipeline(questions, retrieve_fn, label="dense", n=100):
    """Run retrieval + LLM on first n questions. Returns predictions dict."""
    predictions = {}
    for i, q in enumerate(questions[:n]):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)
        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ])
        pred = parse_answer(raw)
        predictions[q["id"]] = pred if pred else 1  # default to 1 if parse fails
        print(f"  Q{q['id']:>3}: pred={predictions[q['id']]}")
        time.sleep(0.3)  # be nice to the API
    print(f"\n{label}: answered {len(predictions)} questions")
    return predictions

# Dense retrieval function
def dense_retrieve_chunks(query):
    idx, _ = dense_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

dense_preds = run_pipeline(questions, dense_retrieve_chunks, label="dense")

  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=9
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=8
  Q  7: pred=1
  Q  8: pred=4
  Q  9: pred=2
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=2
  Q 14: pred=3
  Q 15: pred=7
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=8
  Q 21: pred=3
  Q 22: pred=6
  Q 23: pred=3
  Q 24: pred=3
  Q 25: pred=5
  Q 26: pred=6
  Q 27: pred=2
  Q 28: pred=7
  Q 29: pred=1
  Q 30: pred=3
  Q 31: pred=4
  Q 32: pred=2
  Q 33: pred=8
  Q 34: pred=5
  Q 35: pred=3
  Q 36: pred=2
  Q 37: pred=8
  Q 38: pred=6
  Q 39: pred=4
  Q 40: pred=1
  Q 41: pred=1
  Q 42: pred=2
  Q 43: pred=4
  Q 44: pred=1
  Q 45: pred=2
  Q 46: pred=1
  Q 47: pred=2
  Q 48: pred=2
  Q 49: pred=6
  Q 50: pred=5
  Q 51: pred=7
  Q 52: pred=4
  Q 53: pred=9
  Q 54: pred=9
  Q 55: pred=9
  Q 56: pred=9
  Q 57: pred=9
  Q 58: pred=9
  Q 59: pred=9
  Q 60: pred=9
  Q 61: pred=9
  Q 62: pred=9
  Q 63: pred=9
  Q 64: pred=5
  Q 65: pred=3
  Q 66: pred=7
  Q 67: pr

---
## Section 2: Sparse Retrieval (BM25)

**BM25** is a keyword-matching algorithm. It scores documents by how many query terms they contain, weighted by term rarity (IDF). No neural network needed.

### 2.1 Thai Tokenization

BM25 needs tokenized text. Thai has no spaces between words, so we use `pythainlp` to segment.

In [ ]:
from pythainlp.tokenize import word_tokenize

# Demo: tokenize a Thai sentence
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"
tokens = word_tokenize(sample, engine="newmm")
print(f"Input:  {sample}")
print(f"Tokens: {tokens}")

Input:  Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens: ['Watch', ' ', 'S', '3', ' ', 'Ultra', ' ', 'กันน้ำ', 'ได้', 'กี่', ' ', 'ATM', ' ', 'ครับ', '?']


### 2.2 Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize all chunks
tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks")

BM25 index built over 1055 chunks


### 2.3 Retrieve with BM25

Compare BM25 results with dense results for the same question.

In [ ]:
def bm25_retrieve(query, k=TOP_K):
    """Return top-k chunk indices using BM25."""
    tokens = word_tokenize(query, engine="newmm")
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Compare: same question, two retrieval methods
q = questions[0]
print(f"Question: {q['question']}\n")

d_idx, _ = dense_retrieve(q["question"], chunk_embeddings)
b_idx, _ = bm25_retrieve(q["question"])

print("Dense top-5 sources:")
for i in d_idx:
    print(f"  {chunks[i]['source']}")

print("\nBM25 top-5 sources:")
for i in b_idx:
    print(f"  {chunks[i]['source']}")

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Dense top-5 sources:
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-004_wongkhojon_watch_s3_se.md
  products/WK-SW-003_wongkhojon_watch_s3.md

BM25 top-5 sources:
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md


### 2.4 Run All Questions (BM25)

In [ ]:
def bm25_retrieve_chunks(query):
    idx, _ = bm25_retrieve(query)
    return [chunks[i] for i in idx]

bm25_preds = run_pipeline(questions, bm25_retrieve_chunks, label="bm25")

  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=9
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=8
  Q  7: pred=1
  Q  8: pred=1
  Q  9: pred=2
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=6
  Q 14: pred=3
  Q 15: pred=4
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=3
  Q 21: pred=3
  Q 22: pred=6
  Q 23: pred=3
  Q 24: pred=3
  Q 25: pred=5
  Q 26: pred=6
  Q 27: pred=2
  Q 28: pred=7
  Q 29: pred=4
  Q 30: pred=3
  Q 31: pred=4
  Q 32: pred=2
  Q 33: pred=1
  Q 34: pred=5
  Q 35: pred=3
  Q 36: pred=2
  Q 37: pred=8
  Q 38: pred=6
  Q 39: pred=4
  Q 40: pred=4
  Q 41: pred=7
  Q 42: pred=2
  Q 43: pred=4
  Q 44: pred=1
  Q 45: pred=2
  Q 46: pred=1
  Q 47: pred=2
  Q 48: pred=8
  Q 49: pred=6
  Q 50: pred=5
  Q 51: pred=7
  Q 52: pred=4
  Q 53: pred=9
  Q 54: pred=9
  Q 55: pred=9
  Q 56: pred=3
  Q 57: pred=9
  Q 58: pred=9
  Q 59: pred=9
  Q 60: pred=9
  Q 61: pred=9
  Q 62: pred=9
  Q 63: pred=9
  Q 64: pred=5
  Q 65: pred=3
  Q 66: pred=7
  Q 67: pr

---
## Section 3: Hybrid Retrieval (RRF)

**Hybrid** combines dense and sparse results. The idea: dense is good at semantic matching, BM25 is good at exact keyword matching. Together they cover more cases.

We use **Reciprocal Rank Fusion (RRF)** to merge the two ranked lists:

$$\text{score}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60). Documents ranked highly by *either* method get a high combined score.

In [ ]:
def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60):
    """Combine dense + BM25 results using Reciprocal Rank Fusion."""
    # Get top candidates from each method (fetch more than k to improve fusion)
    fetch_k = k * 2
    d_idx, _ = dense_retrieve(query, chunk_embs, k=fetch_k)
    b_idx, _ = bm25_retrieve(query, k=fetch_k)

    # Compute RRF scores
    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    # Sort by combined score, return top-k
    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

# Demo
q = questions[0]
h_idx = hybrid_retrieve(q["question"], chunk_embeddings)
print(f"Question: {q['question']}\n")
print("Hybrid top-5 sources:")
for i in h_idx:
    print(f"  {chunks[i]['source']}")

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Hybrid top-5 sources:
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-003_wongkhojon_watch_s3.md


### 3.2 Run All Questions (Hybrid)

In [ ]:
def hybrid_retrieve_chunks(query):
    idx = hybrid_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

hybrid_preds = run_pipeline(questions, hybrid_retrieve_chunks, label="hybrid")

  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=5
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=8
  Q  7: pred=1
  Q  8: pred=4
  Q  9: pred=2
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=6
  Q 14: pred=3
  Q 15: pred=7
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=2
  Q 21: pred=3
  Q 22: pred=3
  Q 23: pred=3
  Q 24: pred=3
  Q 25: pred=2
  Q 26: pred=6
  Q 27: pred=2
  Q 28: pred=7
  Q 29: pred=4
  Q 30: pred=3
  Q 31: pred=4
  Q 32: pred=2
  Q 33: pred=8
  Q 34: pred=5
  Q 35: pred=3
  Q 36: pred=2
  Q 37: pred=8
  Q 38: pred=6
  Q 39: pred=4
  Q 40: pred=1
  Q 41: pred=7
  Q 42: pred=2
  Q 43: pred=4
  Q 44: pred=1
  Q 45: pred=2
  Q 46: pred=1
  Q 47: pred=2
  Q 48: pred=2
  Q 49: pred=6
  Q 50: pred=5
  Q 51: pred=7
  Q 52: pred=4
  Q 53: pred=9
  Q 54: pred=9
  Q 55: pred=9
  Q 56: pred=9
  Q 57: pred=9
  Q 58: pred=9
  Q 59: pred=9
  Q 60: pred=9
  Q 61: pred=9
  Q 62: pred=9
  Q 63: pred=9
  Q 64: pred=5
  Q 65: pred=3
  Q 66: pred=7
  Q 67: pr

### 3.3 Compare All Three Methods

In [ ]:
print(f"{'QID':>4}  {'Dense':>6} {'BM25':>6} {'Hybrid':>7}")
print("-" * 30)
for q in questions[:N_QUESTIONS]:
    qid = q["id"]
    d = dense_preds.get(qid, "-")
    b = bm25_preds.get(qid, "-")
    h = hybrid_preds.get(qid, "-")
    match = "" if d == b == h else "  ← differ"
    print(f"Q{qid:>3}  {d:>6} {b:>6} {h:>7}{match}")

 QID   Dense   BM25  Hybrid
------------------------------
Q  1       5      5       5
Q  2       7      7       7
Q  3       9      9       5  ← differ
Q  4       6      6       6
Q  5       6      6       6
Q  6       8      8       8
Q  7       1      1       1
Q  8       4      1       4  ← differ
Q  9       2      2       2
Q 10       2      2       2


### Write Submission

Pick your best method and generate a `submission.csv` for Kaggle.

> Set `N_QUESTIONS = 100` at the top and re-run the notebook to generate a full submission.

# Dense Prediction

In [ ]:
# Use dense predictions as the submission (change to hybrid_preds or bm25_preds to try others)
best_preds = dense_preds

with open("submission_dense_rag.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, best_preds.get(qid, 1)])  # default=1 for unanswered

print(f"submission_dense_rag.csv written ({len(questions)} rows)")

submission.csv written (100 rows)


# BM25 Prediction

In [ ]:
# Use dense predictions as the submission (change to hybrid_preds or bm25_preds to try others)
best_preds = bm25_preds

with open("submission_bm25_rag.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, best_preds.get(qid, 1)])  # default=1 for unanswered

print(f"submission_bm25_rag.csv written ({len(questions)} rows)")

submission.csv written (100 rows)


# RRF Prediction

In [ ]:
# Use dense predictions as the submission (change to hybrid_preds or bm25_preds to try others)
best_preds = hybrid_preds

with open("submission_hybrid_preds_rag.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, best_preds.get(qid, 1)])  # default=1 for unanswered

print(f"submission_hybrid_preds_rag.csv written ({len(questions)} rows)")

submission_hybrid_preds_rag.csv written (100 rows)


---
## Section 4: What's Next?

This baseline uses a simple setup. Here are ideas to improve your score:

- **Better embeddings** — try other, stronger multilingual  embedding.
- **Smarter chunking** — split by structure or other methods or add useful information to each chunk
- **Chunk size tuning** — experiment with  256, 512, 1024 or something else character chunks
- **Different ThaiLLMs** — try `openthaigpt`, `kbtg`, `pathumma`.
- **Prompt engineering** — adjust the system prompt, add few-shot examples, or change the output format
- **Reranking** — use a cross-encoder or specialized reranker to re-score the top-k chunks before sending to the

**Feel free to implement your own RAG.**